In [1]:
# Import all required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image, ImageTk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
import warnings
import tkinter as tk
from tkinter import filedialog, messagebox
import threading
import pyttsx3

# Settings and configurations
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# Define paths and parameters (UPDATE THESE PATHS)
train_dir = "E:/Sem 6/Machine Learning/Final Project/dataset/Snake Images/train"
test_dir = "E:/Sem 6/Machine Learning/Final Project/dataset/Snake Images/test"
batch_size = 32
epochs = 10
threshold = 0.5  # Lowered threshold for better non-venomous detection

In [3]:
# Define image transformations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
# Load datasets
train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
test_data = datasets.ImageFolder(test_dir, transform=test_transforms)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
class_names = train_data.classes

# Initialize model
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

print("Class names:", class_names)  # Verify your class names

Class names: ['Non Venomous', 'Venomous']


In [5]:
def train_model():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    
    print("Starting training...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")
    print("Training complete!")

In [6]:
def predict_image_with_tta(image_path, model, base_transform, class_names, threshold=0.5):
    model.eval()
    try:
        image = Image.open(image_path).convert('RGB')
        
        augmentations = [
            base_transform,
            transforms.Compose([
                transforms.RandomHorizontalFlip(p=1.0),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ]),
            transforms.Compose([
                transforms.RandomRotation(15),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        ]

        probs = []
        for transform in augmentations:
            img_tensor = transform(image).unsqueeze(0).to(device)
            with torch.no_grad():
                output = model(img_tensor)
                prob = F.softmax(output, dim=1).cpu().numpy()
                probs.append(prob[0])

        avg_probs = np.mean(probs, axis=0)
        predicted_class = np.argmax(avg_probs)
        confidence = avg_probs[predicted_class]

        # Debug output
        print("\nPrediction Details:")
        print(f"Class probabilities: {dict(zip(class_names, avg_probs))}")
        print(f"Predicted class: {class_names[predicted_class]}")
        print(f"Confidence: {confidence:.4f}")

        if confidence < threshold:
            return "Uncertain Prediction", confidence, None
        else:
            return class_names[predicted_class], confidence, class_names[predicted_class]
    except Exception as e:
        print(f"Prediction error: {e}")
        return "Error in prediction", 0.0, None

In [7]:
class SnakeClassifierApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Snake Classifier")
        self.root.geometry("500x600")
        
        self.model = model
        self.class_names = class_names
        self.threshold = threshold
        self.test_transforms = test_transforms
        
        self.create_widgets()
        
    def create_widgets(self):
        # Button to select image
        self.btn = tk.Button(self.root, text="Choose Image", command=self.select_image, 
                            font=("Arial", 14), bg="lightblue")
        self.btn.pack(pady=20)
        
        # Label to display image
        self.image_label = tk.Label(self.root)
        self.image_label.pack()
        
        # Label to display result
        self.result_label = tk.Label(self.root, text="Prediction will appear here", 
                                   font=("Arial", 14))
        self.result_label.pack(pady=10)
        
        # Label to display alert message
        self.alert_label = tk.Label(self.root, text="", font=("Arial", 12, "bold"))
        self.alert_label.pack(pady=5)
        
        # Training button
        self.train_btn = tk.Button(self.root, text="Train Model", command=self.train_model,
                                 font=("Arial", 12), bg="lightgreen")
        self.train_btn.pack(pady=10)
        
    def train_model(self):
        self.btn.config(state=tk.DISABLED)
        self.train_btn.config(state=tk.DISABLED)
        threading.Thread(target=self._train_model_thread).start()
        
    def _train_model_thread(self):
        train_model()
        self.btn.config(state=tk.NORMAL)
        self.train_btn.config(state=tk.NORMAL)
        messagebox.showinfo("Training Complete", "Model training finished!")
        
    def select_image(self):
        file_path = filedialog.askopenfilename()
        if file_path:
            image = Image.open(file_path).resize((300, 300))
            photo = ImageTk.PhotoImage(image)
            self.image_label.config(image=photo)
            self.image_label.image = photo
            threading.Thread(target=self.run_prediction, args=(file_path,)).start()
    
    def run_prediction(self, image_path):
        result, confidence, clean_result = predict_image_with_tta(
            image_path, self.model, self.test_transforms, self.class_names, self.threshold)
        self.root.after(0, self._update_prediction, result, confidence, clean_result)
    
    def _update_prediction(self, result, confidence, clean_result):
        # Display prediction result
        self.result_label.config(text=f"Prediction: {result} ({confidence:.2f})")
        
        # Determine alert message
        if clean_result:
            # Normalize the string for comparison
            normalized = clean_result.lower().replace(" ", "").replace("-", "").strip()
            
            # Check for non-venomous first (order matters!)
            if "nonvenomous" in normalized:
                alert_msg = "Non-venomous snake detected\nNo immediate danger but avoid contact"
                color = "green"
            elif "venomous" in normalized:
                alert_msg = "⚠️ DANGER! Venomous Snake Detected! ⚠️\nSeek immediate medical attention!"
                color = "red"
            else:
                alert_msg = f"Identified as: {clean_result}\nApproach with caution"
                color = "blue"
        else:
            alert_msg = "⚠️ Uncertain Snake Type ⚠️\nAssume venomous and keep safe distance"
            color = "orange"
        
        # Update UI
        self.result_label.config(fg=color)
        self.alert_label.config(text=alert_msg, fg=color)
        
        # Text-to-speech
        try:
            engine = pyttsx3.init()
            if color == "red":  # Venomous
                engine.setProperty('rate', 150)
                engine.setProperty('volume', 1.0)
                voices = engine.getProperty('voices')
                engine.setProperty('voice', voices[0].id)  # Typically a deeper voice
            else:
                engine.setProperty('rate', 130)
                engine.setProperty('volume', 0.8)
            engine.say(alert_msg)
            engine.runAndWait()
        except Exception as e:
            print(f"TTS error: {e}")

In [8]:
if __name__ == "__main__":
    root = tk.Tk()
    app = SnakeClassifierApp(root)
    root.mainloop()

Starting training...
Epoch 1/10, Loss: 0.5752
Epoch 2/10, Loss: 0.3288
Epoch 3/10, Loss: 0.2060
Epoch 4/10, Loss: 0.1310
Epoch 5/10, Loss: 0.0830
Epoch 6/10, Loss: 0.0581
Epoch 7/10, Loss: 0.0466
Epoch 8/10, Loss: 0.0348
Epoch 9/10, Loss: 0.0389
Epoch 10/10, Loss: 0.0420
Training complete!

Prediction Details:
Class probabilities: {'Non Venomous': np.float32(0.0023166), 'Venomous': np.float32(0.99768335)}
Predicted class: Venomous
Confidence: 0.9977
Starting training...
Epoch 1/10, Loss: 0.0181
Epoch 2/10, Loss: 0.0284
